In [1]:
import warnings

warnings.filterwarnings("ignore")

# protobuf compatibility shim for TensorFlow 2.18 with protobuf 6+
try:
    from google.protobuf import message_factory

    if not hasattr(message_factory.MessageFactory, "GetPrototype"):

        def GetPrototype(self, descriptor):
            # Fallback to the newer GetMessageClass method
            return self.GetMessageClass(descriptor)

        message_factory.MessageFactory.GetPrototype = GetPrototype
except Exception:
    pass

import pandas as pd, numpy as np, json, gc, random, os, sys
from matplotlib import pyplot as plt
from tqdm import tqdm

import tensorflow as tf

import tensorflow.keras.backend as K
import tensorflow.keras.layers as L

from sklearn.model_selection import train_test_split, KFold




2026-05-08 23:51:13.653419: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778284273.666887      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778284273.671265      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-08 23:51:13.688320: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
train = pd.read_json("/kaggle/input/stanford-covid-vaccine/train.json", lines=True)
test = pd.read_json("/kaggle/input/stanford-covid-vaccine/test.json", lines=True)
sample_sub = pd.read_csv("/kaggle/input/stanford-covid-vaccine/sample_submission.csv")




In [3]:
print(train.shape)
if not train.isnull().values.any():
    print("No missing values")
train.head()




(2160, 19)
No missing values


,index,id,sequence,structure,predicted_loop_type,signal_to_noise,SN_filter,seq_length,seq_scored,reactivity_error,deg_error_Mg_pH10,deg_error_pH10,deg_error_Mg_50C,deg_error_50C,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,0,id_001f94081,GGAAAAGCUCUAAUAACAGGAGACUAGGACUACGUAUUUCUAGGUA...,.....((((((.......)))).)).((.....((..((((((......,EEEEESSSSSSHHHHHHHSSSSBSSXSSIIIIISSIISSSSSSHHH...,6.894,1,107,68,"[0.1359, 0.20700000000000002, 0.1633, 0.1452, ...","[0.26130000000000003, 0.38420000000000004, 0.1...","[0.2631, 0.28600000000000003, 0.0964, 0.1574, ...","[0.1501, 0.275, 0.0947, 0.18660000000000002, 0...","[0.2167, 0.34750000000000003, 0.188, 0.2124, 0...","[0.3297, 1.5693000000000001, 1.1227, 0.8686, 0...","[0.7556, 2.983, 0.2526, 1.3789, 0.637600000000...","[2.3375, 3.5060000000000002, 0.3008, 1.0108, 0...","[0.35810000000000003, 2.9683, 0.2589, 1.4552, ...","[0.6382, 3.4773, 0.9988, 1.3228, 0.78770000000..."
1,1,id_0049f53ba,GGAAAAAGCGCGCGCGGUUAGCGCGCGCUUUUGCGCGCGCUGUACC...,.....(((((((((((((((((((((((....)))))))))).)))...,EEEEESSSSSSSSSSSSSSSSSSSSSSSHHHHSSSSSSSSSSBSSS...,0.193,0,107,68,"[2.8272, 2.8272, 2.8272, 4.7343, 2.5676, 2.567...","[73705.3985, 73705.3985, 73705.3985, 73705.398...","[10.1986, 9.2418, 5.0933, 5.0933, 5.0933, 5.09...","[16.6174, 13.868, 8.1968, 8.1968, 8.1968, 8.19...","[15.4857, 7.9596, 13.3957, 5.8777, 5.8777, 5.8...","[0.0, 0.0, 0.0, 2.2965, 0.0, 0.0, 0.0, 0.0, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[4.947, 4.4523, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[4.8511, 4.0426, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[7.6692, 0.0, 10.9561, 0.0, 0.0, 0.0, 0.0, 0.0..."
2,2,id_006f36f57,GGAAAGUGCUCAGAUAAGCUAAGCUCGAAUAGCAAUCGAAUAGAAU...,.....((((.((.....((((.(((.....)))..((((......)...,EEEEESSSSISSIIIIISSSSMSSSHHHHHSSSMMSSSSHHHHHHS...,8.800,1,107,68,"[0.0931, 0.13290000000000002, 0.11280000000000...","[0.1365, 0.2237, 0.1812, 0.1333, 0.1148, 0.160...","[0.17020000000000002, 0.178, 0.111, 0.091, 0.0...","[0.1033, 0.1464, 0.1126, 0.09620000000000001, ...","[0.14980000000000002, 0.1761, 0.1517, 0.116700...","[0.44820000000000004, 1.4822, 1.1819, 0.743400...","[0.2504, 1.4021, 0.9804, 0.49670000000000003, ...","[2.243, 2.9361, 1.0553, 0.721, 0.6396000000000...","[0.5163, 1.6823000000000001, 1.0426, 0.7902, 0...","[0.9501000000000001, 1.7974999999999999, 1.499..."
3,3,id_0082d463b,GGAAAAGCGCGCGCGCGCGCGCGAAAAAGCGCGCGCGCGCGCGCGC...,......((((((((((((((((......))))))))))))))))((...,EEEEEESSSSSSSSSSSSSSSSHHHHHHSSSSSSSSSSSSSSSSSS...,0.104,0,107,68,"[3.5229, 6.0748, 3.0374, 3.0374, 3.0374, 3.037...","[73705.3985, 73705.3985, 73705.3985, 73705.398...","[11.8007, 12.7566, 5.7733, 5.7733, 5.7733, 5.7...","[121286.7181, 121286.7182, 121286.7181, 121286...","[15.3995, 8.1124, 7.7824, 7.7824, 7.7824, 7.78...","[0.0, 2.2399, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0....","[0.0, -0.5083, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0...","[3.4248, 6.8128, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, -0.8365, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0...","[7.6692, -1.3223, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0..."
4,4,id_0087940f4,GGAAAAUAUAUAAUAUAUUAUAUAAAUAUAUUAUAGAAGUAUAAUA...,.....(((((((.((((((((((((.(((((((((....)))))))...,EEEEESSSSSSSBSSSSSSSSSSSSBSSSSSSSSSHHHHSSSSSSS...,0.423,0,107,68,"[1.665, 2.1728, 2.0041, 1.2405, 0.620200000000...","[4.2139, 3.9637000000000002, 3.2467, 2.4716, 1...","[3.0942, 3.015, 2.1212, 2.0552, 0.881500000000...","[2.6717, 2.4818, 1.9919, 2.5484999999999998, 1...","[1.3285, 3.6173, 1.3057, 1.3021, 1.1507, 1.150...","[0.8267, 2.6577, 2.8481, 0.40090000000000003, ...","[2.1058, 3.138, 2.5437000000000003, 1.0932, 0....","[4.7366, 4.6243, 1.2068, 1.1538, 0.0, 0.0, 0.7...","[2.2052, 1.7947000000000002, 0.7457, 3.1233, 0...","[0.0, 5.1198, -0.3551, -0.3518, 0.0, 0.0, 0.0,..."


In [4]:
print(test.shape)
if not test.isnull().values.any():
    print("No missing values")
test.head()




(240, 7)
No missing values


,index,id,sequence,structure,predicted_loop_type,seq_length,seq_scored
0,0,id_00b436dec,GGAAAUCAUCGAGGACGGGUCCGUUCAGCACGCGAAAGCGUCGUGA...,.....(((((((((((..(((((((((..((((....))))..)))...,EEEEESSSSSSSSSSSIISSSSSSSSSIISSSSHHHHSSSSIISSS...,107,68
1,1,id_010ab0472,GGAAAGCAUGGGACCACGAUUCACAUCGGUCUGCACGUAGGACAUU...,.....(((...((((..(((....))))))))))(((((((((......,EEEEESSSBBBSSSSBBSSSHHHHSSSSSSSSSSSSSSSSSSSIII...,107,68
2,2,id_01ccf95c0,GGAAAAGCGAACGACGAAACGCGGGCGCGAUGGACAGGAGGCUGAC...,......(((..((.((...))))..)))((((..(((...((((.....,EEEEEESSSIISSBSSHHHSSSSIISSSSSSSIISSSIIISSSSHH...,107,68
3,3,id_023f91e97,GGAAAUUAUAAUGAUCAUGAGGUACUAUCAUCAUGACUGGAGACAG...,..........(((.((.((..((.((.((.((.((.(((....)))...,EEEEEEEEEESSSBSSBSSIISSBSSBSSBSSBSSBSSSHHHHSSS...,107,68
4,4,id_0295caf90,GGAAACAGGGCAGGUCGAGUGAGGGAUCGAGGCGGGCGCGGUUCGG...,.........(((.(((..(((....((((((.((....)).)))))...,EEEEEEEEESSSISSSIISSSIIIISSSSSSISSHHHHSSISSSSS...,107,68


In [5]:
print(sample_sub.shape)
if not sample_sub.isnull().values.any():
    print("No missing values")
sample_sub.head()




(25680, 6)
No missing values


,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.0,0.0,0.0,0.0,0.0
1,id_00b436dec_1,0.0,0.0,0.0,0.0,0.0
2,id_00b436dec_2,0.0,0.0,0.0,0.0,0.0
3,id_00b436dec_3,0.0,0.0,0.0,0.0,0.0
4,id_00b436dec_4,0.0,0.0,0.0,0.0,0.0


In [6]:
target_cols = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]




In [7]:
token2int = {x: i for i, x in enumerate("().ACGUBEHIMSX")}




In [8]:
def preprocess_inputs(df, cols=["sequence"]):
    seq_array = np.array(
        df[cols[0]].apply(lambda seq: [token2int[ch] for ch in seq]).tolist()
    )
    return seq_array  # shape: (samples, seq_len)




In [9]:
train_filt = train[train.signal_to_noise > 1]
train_inputs = preprocess_inputs(train_filt)
train_labels = np.array(train_filt[target_cols].values.tolist()).transpose((0, 2, 1))




In [10]:
def gru_layer(hidden_dim, dropout):
    return tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(
            hidden_dim,
            dropout=dropout,
            return_sequences=True,
            kernel_initializer="orthogonal",
        )
    )


def lstm_layer(hidden_dim, dropout):
    return tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(
            hidden_dim,
            dropout=dropout,
            return_sequences=True,
            kernel_initializer="orthogonal",
        )
    )


def build_model(
    gru=False, seq_len=107, pred_len=68, dropout=0.5, embed_dim=75, hidden_dim=128
):
    inputs = tf.keras.layers.Input(shape=(seq_len,))
    embed = tf.keras.layers.Embedding(input_dim=len(token2int), output_dim=embed_dim)(
        inputs
    )
    x = tf.keras.layers.SpatialDropout1D(0.2)(embed)

    if gru:
        x = gru_layer(hidden_dim, dropout)(x)
        x = gru_layer(hidden_dim, dropout)(x)
        x = gru_layer(hidden_dim, dropout)(x)
    else:
        x = lstm_layer(hidden_dim, dropout)(x)
        x = lstm_layer(hidden_dim, dropout)(x)
        x = lstm_layer(hidden_dim, dropout)(x)

    truncated = x[:, :pred_len]  # keep only the scored positions
    out = tf.keras.layers.Dense(5, activation="linear")(truncated)

    model = tf.keras.Model(inputs=inputs, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(), loss="mse")
    return model




In [11]:
train_inputs, val_inputs, train_labels, val_labels = train_test_split(
    train_inputs, train_labels, test_size=0.1, random_state=34
)




In [12]:
if tf.config.list_physical_devices("GPU"):
    print("Training on GPU")
else:
    print("Training on CPU")




Training on GPU


In [13]:
lr_callback = tf.keras.callbacks.ReduceLROnPlateau()




In [14]:
gru = build_model(
    gru=True, seq_len=train_inputs.shape[1], pred_len=train_labels.shape[1]
)
sv_gru = tf.keras.callbacks.ModelCheckpoint("model_gru.h5", save_best_only=True)

history_gru = gru.fit(
    train_inputs,
    train_labels,
    validation_data=(val_inputs, val_labels),
    batch_size=64,
    epochs=5,  # reduced for quick execution
    callbacks=[lr_callback, sv_gru],
    verbose=2,
)

print(
    f"GRU: min train loss={min(history_gru.history['loss'])}, min val loss={min(history_gru.history['val_loss'])}"
)




2026-05-08 23:51:20.747937: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778284280.748396      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6


Epoch 1/5


I0000 00:00:1778284285.102578     107 cuda_dnn.cc:529] Loaded cuDNN version 90300


27/27 - 5s - 194ms/step - loss: 0.2784 - val_loss: 0.2219 - learning_rate: 0.0010
Epoch 2/5


27/27 - 1s - 28ms/step - loss: 0.2202 - val_loss: 0.1878 - learning_rate: 0.0010
Epoch 3/5


27/27 - 1s - 26ms/step - loss: 0.1950 - val_loss: 0.1778 - learning_rate: 0.0010
Epoch 4/5


27/27 - 1s - 30ms/step - loss: 0.1798 - val_loss: 0.1580 - learning_rate: 0.0010
Epoch 5/5


27/27 - 1s - 26ms/step - loss: 0.1667 - val_loss: 0.1498 - learning_rate: 0.0010
GRU: min train loss=0.16667553782463074, min val loss=0.1498284637928009


In [15]:
lstm = build_model(
    gru=False, seq_len=train_inputs.shape[1], pred_len=train_labels.shape[1]
)
sv_lstm = tf.keras.callbacks.ModelCheckpoint("model_lstm.h5", save_best_only=True)

history_lstm = lstm.fit(
    train_inputs,
    train_labels,
    validation_data=(val_inputs, val_labels),
    batch_size=64,
    epochs=5,
    callbacks=[lr_callback, sv_lstm],
    verbose=2,
)

print(
    f"LSTM: min train loss={min(history_lstm.history['loss'])}, min val loss={min(history_lstm.history['val_loss'])}"
)




Epoch 1/5


27/27 - 4s - 155ms/step - loss: 0.2930 - val_loss: 0.2300 - learning_rate: 0.0010
Epoch 2/5


27/27 - 1s - 27ms/step - loss: 0.2188 - val_loss: 0.1929 - learning_rate: 0.0010
Epoch 3/5


27/27 - 1s - 28ms/step - loss: 0.1950 - val_loss: 0.1747 - learning_rate: 0.0010
Epoch 4/5


27/27 - 1s - 29ms/step - loss: 0.1832 - val_loss: 0.1678 - learning_rate: 0.0010
Epoch 5/5


27/27 - 1s - 32ms/step - loss: 0.1774 - val_loss: 0.1630 - learning_rate: 0.0010
LSTM: min train loss=0.1773853898048401, min val loss=0.16299189627170563


In [16]:
public_df = test.query("seq_length == 107").copy()
private_df = test.query("seq_length == 130").copy()

public_inputs = preprocess_inputs(public_df)
private_inputs = (
    preprocess_inputs(private_df)
    if not private_df.empty
    else np.empty((0, 0), dtype=int)
)




In [17]:
gru_short = build_model(gru=True, seq_len=107, pred_len=68)
gru_long = build_model(gru=True, seq_len=130, pred_len=68)
lstm_short = build_model(gru=False, seq_len=107, pred_len=68)
lstm_long = build_model(gru=False, seq_len=130, pred_len=68)

for mdl, path in [
    (gru_short, "model_gru.h5"),
    (gru_long, "model_gru.h5"),
    (lstm_short, "model_lstm.h5"),
    (lstm_long, "model_lstm.h5"),
]:
    if os.path.exists(path):
        mdl.load_weights(path)


gru_public_preds = gru_short.predict(public_inputs, verbose=0)
gru_private_preds = (
    gru_long.predict(private_inputs, verbose=0)
    if private_inputs.size
    else np.empty((0, 68, 5))
)
lstm_public_preds = lstm_short.predict(public_inputs, verbose=0)
lstm_private_preds = (
    lstm_long.predict(private_inputs, verbose=0)
    if private_inputs.size
    else np.empty((0, 68, 5))
)




In [18]:
preds_gru = []
for df, preds in [(public_df, gru_public_preds), (private_df, gru_private_preds)]:
    if preds.shape[0] == 0:
        continue
    for i, uid in enumerate(df.id):
        single_pred = preds[i]  # shape (pred_len, 5)
        single_df = pd.DataFrame(single_pred, columns=target_cols)
        single_df["id_seqpos"] = [f"{uid}_{x}" for x in range(single_df.shape[0])]
        preds_gru.append(single_df)

preds_gru_df = pd.concat(preds_gru, ignore_index=True)




In [19]:
preds_lstm = []
for df, preds in [(public_df, lstm_public_preds), (private_df, lstm_private_preds)]:
    if preds.shape[0] == 0:
        continue
    for i, uid in enumerate(df.id):
        single_pred = preds[i]
        single_df = pd.DataFrame(single_pred, columns=target_cols)
        single_df["id_seqpos"] = [f"{uid}_{x}" for x in range(single_df.shape[0])]
        preds_lstm.append(single_df)

preds_lstm_df = pd.concat(preds_lstm, ignore_index=True)




In [20]:
blend_preds_df = pd.DataFrame()
blend_preds_df["id_seqpos"] = preds_gru_df["id_seqpos"]
for col in target_cols:
    blend_preds_df[col] = 0.5 * preds_gru_df[col] + 0.5 * preds_lstm_df[col]




In [21]:
submission = sample_sub[["id_seqpos"]].merge(blend_preds_df, on="id_seqpos", how="left")
submission[target_cols] = submission[target_cols].fillna(0)
submission.head()




,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.832239,1.017252,2.314584,0.994758,1.072720
1,id_00b436dec_1,1.392224,1.605345,2.515584,1.631543,1.600942
2,id_00b436dec_2,1.372744,1.149630,1.395811,1.471951,1.436796
3,id_00b436dec_3,1.079532,1.011814,1.033776,1.234632,1.162855
4,id_00b436dec_4,0.821916,0.877112,0.843389,1.003333,0.952340


In [22]:
submission.to_csv("submission.csv", index=False)
print("Submission saved")

Submission saved
